# Generate evaluation dataset to evaluate LLMs
- all input data is in the `Data` directory, or available on Github
- Unzip them


### Import libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import io

### Construct data from the CIVIC db
- At the heart of CIViC is the clinical evidence statement. The clinical evidence statement is a piece of information that has been manually curated from trustable medical literature about a molecular profile (variant) or genomic ‘event’ that has implications for protein function, oncogenicity, cancer predisposition, diagnosis (aka molecular classification), prognosis, or predictive response to therapy. For example, “Patients with BRAF V600 mutations respond well to the drug dabrafenib”. A molecular profile is comprised of one or more variant(s) which may be a single nucleotide substitution, a small insertion or deletion, an RNA gene fusion, a chromosomal rearrangement, an RNA expression pattern (e.g. over-expression), etc. Each clinical evidence statement corresponds to a single citable publication.
- Evidence Items follow a structured knowledge model with required fields: Molecular Profile Name (Gene/Variant), Source, Variant Origin, Disease, Evidence Statement, Evidence Type, Evidence Level, Evidence Direction, Significance, and Trust Rating) with additional optional fields (Associated Phenotypes, etc). For some Evidence Types, additional required or optional fields become available (e.g., Predictive Evidence Types require a Therapy/Drug Name).

In [ ]:
# https://civicdb.org/downloads/01-Jun-2025/01-Jun-2025-ClinicalEvidenceSummaries.tsv
# gunzip input in Data
evidence_info = pd.read_csv('Data/01-Jun-2025-ClinicalEvidenceSummaries.tsv', sep='\t')

In [ ]:
evidence_info.head()

In [ ]:
evidence_info.columns

In [ ]:
evidence_info.shape

In [ ]:
evidence_info['evidence_status'].unique()

In [ ]:
evidence_info['evidence_level'].unique()

In [ ]:
evidence_info['evidence_level'].value_counts()

In [ ]:
# exclude fusion and frameshift mutations (frameshifts are hard to match with GDC annotations)
ssms_df = evidence_info[
     (~evidence_info['molecular_profile'].str.contains('Fusion')) & \
     (~evidence_info['molecular_profile'].str.contains('FS'))
    ]

In [ ]:
ssms_df.shape

In [ ]:
ssms_df['molecular_profile'].unique()

In [ ]:
ssms_df.drop_duplicates(['molecular_profile'], inplace=True)

In [ ]:
ssms_df.shape

### combination mutation data
- how many ssm + ssm
- ssm + cnv
- cnv + cnv

In [ ]:
ssms_df[ssms_df['molecular_profile'].str.contains('AND')].head()

### decompose molecular profile into:
- gene
- mutation
- disease

In [ ]:
ssms_df_copy = ssms_df.copy()
ssms_df_copy['gene'] = ssms_df_copy['molecular_profile'].str.split(' ').str[0]
ssms_df_copy['mutation'] = ssms_df_copy['molecular_profile'].str.split(' ').str[1]

In [ ]:
ssms_df_copy.shape

In [ ]:
ssms_df_copy.head()

### check if a mutation is an ssm
- add an `is_ssm` flag

In [ ]:
ssms_df_copy['is_ssm'] = ssms_df_copy['mutation'].str.contains(r'[A-Z]\d+[A-Z]')

In [ ]:
ssms_df_copy['is_ssm'].value_counts()

In [ ]:
ssms_df_copy[ssms_df_copy['is_ssm'] == True].shape

In [ ]:
ssms_df_copy[ssms_df_copy['is_ssm'] == True].head(n=3)

In [ ]:
data_for_eval = ssms_df_copy[ssms_df_copy['is_ssm'] == True]

In [ ]:
data_for_eval.shape

In [ ]:
data_for_eval.head(n=6)

In [ ]:
data_exclude_fusions = data_for_eval[~data_for_eval.molecular_profile.str.contains('Fusion') &
                                     ~data_for_eval.molecular_profile.str.contains(':') &
                                     ~data_for_eval.molecular_profile.str.contains('/')
                                     ]

In [ ]:
data_exclude_fusions.shape

In [ ]:
len(data_exclude_fusions['gene'].unique())

In [ ]:
data_exclude_fusions.columns

In [ ]:
new_mapping = {
    'gene': 'gene_x',
    'mutation': 'mutation_x'
}
new_colnames = [ new_mapping[col] if col in new_mapping  else col for col in data_exclude_fusions.columns ]

In [ ]:
new_colnames[-5:-1]

In [ ]:
data_exclude_fusions.columns = new_colnames

In [ ]:
dataset1 = data_exclude_fusions[['gene_x', 'mutation_x', 'disease', 'molecular_profile']]

In [ ]:
dataset1.to_csv('molecular_profile_civic.csv')

### define template questions

In [ ]:
# generate synthetic data for different types of questions
# note these are for single mutations
# for combination mutations, use hand-curated examples from Cheng et al

template_questions = [
    'What percentage of cancer cases have simple somatic mutations or copy number variants in [gene_x] in the genomic data commons for [disease]?',
    'How often is the [gene_x] [mutation_x] found in [disease] in the genomic data commons?',
    'What is the occurrence rate of [gene_x] [mutation_x] in [disease] in the genomic data commons ?',
    'What is the frequency of simple somatic mutations or copy number variants in [gene_x] in [disease] in the genomic data commons?',
    'What is the incidence of simple somatic mutations or copy number variants in [gene_x] in the genomic data commons for [disease] ?',
    'What is the frequency of [gene_x] [mutation_x] in [disease] in the genomic data commons?',
    'How common are simple somatic mutations or copy number variants in [gene_x] in [disease] in the genomic data commons?',
    'What fraction of cases have simple somatic mutations or copy number variants in [gene_x] in [disease] in the genomic data commons?',
    'What is the rate of occurrence of [gene_x] [mutation_x] mutation in [disease] in the genomic data commons?',
    'How frequently are [gene_x] [mutation_x] mutations detected in [disease] in the genomic data commons?',
    'What proportion of cancer patients exhibit simple somatic mutations or copy number variants in [gene_x] in [disease] in the genomic data commons?',
    'What is the frequency of somatic [gene_x] heterozygous deletion in [disease] in the genomic data commons?',
    'What is the incidence of somatic [gene_x] homozygous deletion in [disease] in the genomic data commons?',
    'Can you provide the frequency of [gene_x] gain in [disease] in the genomic data commons?',
    'In [disease] data from the genomic data commons, what is the frequency of [gene_x] amplification?',
    'What is the frequency of microsatellite instability in [disease] in the genomic data commons?',
    'How common is microsatellite instability in [disease] cases in the genomic data commons?',
    'Can you provide the prevalence of microsatellite instability in [disease] in the genomic data commons?',
    'What percentage of [disease] patients have microsatellite instability in the genomic data commons?',
    'How often is microsatellite instability observed in [disease] in the genomic data commons?',
    'In [disease], what is the occurrence rate of microsatellite instability in the genomic data commons?',
    'What is the incidence of microsatellite instability in [disease] in the genomic data commons?',
]
template_questions_without_mutations = [
    query
    for query in template_questions
    if 'mutation_x' not in query
]


In [ ]:
len(template_questions)

In [ ]:
len(template_questions_without_mutations)

**Synthetic question answer generation**

**Dataset 1**

Generate a simpler and smaller df for first NER test
 - Remove = or * from mutation nomenclature
 - Remove gene/mutation duplicates


In [ ]:
dataset1.head(n=6)

In [ ]:
dataset1.shape

In [ ]:
def generate_q(row):
  # generate question
  mutation_x = row['mutation_x']
  gene_x = row['gene_x']
  disease = row['disease']
  
  if not isinstance(disease, str):
    # fix a generic term if disease not defined
    disease = 'cancer'

  # randomly choose 5 questions from list
  question_list = np.random.choice(template_questions, size = 5, replace=False)
  mod_question_list = []
  for question in question_list:
    try:
      question = question.replace("[mutation_x]", mutation_x).replace("[gene_x]", gene_x).replace("[disease]", disease)
      mod_question_list.append(question)
    except Exception as e:
      # retry with generic cancer disease
      print('unable to generate question')

  return mod_question_list

In [ ]:
dataset1_copy = dataset1.copy()
dataset1_copy['questions'] = dataset1_copy.apply(lambda row: generate_q(row),axis=1)

In [ ]:
dataset1_exploded = dataset1_copy.explode('questions', ignore_index=True)

In [ ]:
dataset1_exploded.shape

In [ ]:
# remove duplicates, retain only unique questions
dataset1_exploded['questions'].nunique()


In [ ]:
dataset1_exploded['questions'].unique()


In [ ]:
dataset1_exploded.head()

In [ ]:
dataset1_exploded_unique = dataset1_exploded.drop_duplicates(subset=['questions'])

In [ ]:
dataset1_exploded_unique.shape

### for combination mutations:
- Use full TSV from civic evidence
- filter for "AND"
- exclude fusions

In [ ]:
combination_mutation_data = ssms_df_copy[(ssms_df_copy.molecular_profile.str.contains('AND')) & \
                                                (ssms_df_copy.molecular_profile.str.contains('Fusion') == False)
                                                ]

In [ ]:
combination_mutation_data.shape

In [ ]:
combination_mutation_data

In [ ]:
dataset1_exploded_unique.head(n=6)

In [ ]:
dataset1_exploded_unique.iloc[0]['questions']

In [ ]:
# run questions from dataset1_exploded_unique through gdc-qag V1 entrypoint to decompose queries into entities, including specific GDC cancer projects
# only retain mutations with frequency of atleast 5% in the GDC
# see https://github.com/uc-cdis/gdc-qag
# results file is dataset3.results.csv

In [ ]:
civic_results = pd.read_csv('Data/dataset3.results.csv', index_col=0)

In [ ]:
# gdc-qag v1 includes decomposed entities: genes, mutations and cancer type
# additionally, it also includes a comparison with and older llama-3b model
civic_results.columns

In [ ]:
civic_results.head(n=3)

In [ ]:
civic_results['mutation_entities'].explode().unique()

### combine with ssms from GDC
- downloaded all ssm freqs from GDC using the mut freq app
- filtered to get atleast 50 cases in >=1 project since we want to exclude 0 frequency

In [ ]:
import pandas as pd
import ast
import numpy as np

In [ ]:
# downloaded all muts from qag-evals github
# https://github.com/uc-cdis/qag-evals/releases/download/mutations-data/frequent-mutations.2025-05-28_all_muts.tsv.gz
# e.g. curl -L -o frequent-mutations.2025-05-28_all_muts.tsv.gz https://github.com/uc-cdis/qag-evals/releases/download/mutations-data/frequent-mutations.2025-05-28_all_muts.tsv.gz
# gunzip input and put in Data folder
mut_freq = pd.read_csv('Data/frequent-mutations.2025-05-28_all_muts.tsv', sep='\t')

In [ ]:
mut_freq.shape

In [ ]:
mut_freq_filtered = mut_freq[mut_freq['num_gdc_ssm_affected_cases'] > 50]

In [ ]:
mut_freq_filtered.shape

In [ ]:
mut_freq_filtered.head(n=3)

In [ ]:
mut_freq_filtered[['gene', 'mutation']] = mut_freq_filtered['protein_change'].str.strip().str.split(expand=True)

In [ ]:
# construct template questions for above mutations
# one question for one project
template_questions = [
    'How often is the [gene_x] [mutation_x] found in [disease] project in the genomic data commons?',
    'What is the occurrence rate of [gene_x] [mutation_x] in [disease] project in the genomic data commons ?',
    'What is the frequency of [gene_x] [mutation_x] in [disease] project in the genomic data commons?',
    'What is the rate of occurrence of [gene_x] [mutation_x] mutation in [disease] project in the genomic data commons?',
    'How frequently are [gene_x] [mutation_x] mutations detected in [disease] project in the genomic data commons?'
]

In [ ]:
project_mappings = pd.read_csv(
  'Data/gdc_projects.tsv', sep='\t', header=None, 
  names=['project', 'desc']
  )
project_mappings['desc'] = project_mappings['desc'].apply(ast.literal_eval)

In [ ]:
project_mappings.iloc[:2]

In [ ]:
np.random.seed(seed=1000)

def generate_ssm_q(row):
  revised_questions = []
  gene = row['gene']
  mutation = row['mutation']
  random_template_q = np.random.choice(template_questions, 1)[0]
  for proj in project_mappings['project']:
    name = project_mappings.loc[project_mappings['project'] == proj]['desc'].values
    disease_desc = name[0][-1]
    disease_full_name = ' '.join([disease_desc, proj])
    try:
      q = random_template_q.replace("[mutation_x]", mutation).replace("[gene_x]", gene).replace("[disease]", disease_full_name)
    except Exception as e:
      q = ''
    revised_questions.append(q)
  
  return revised_questions

In [ ]:
mut_freq_filtered['questions'] = mut_freq_filtered.apply(lambda x: generate_ssm_q(x), axis=1)

In [ ]:
mut_freq_filtered.shape

In [ ]:
mut_freq_filtered.head(n=3).iloc[0]['questions'][:3]

In [ ]:
# 84 mutations x 86 projects
mut_freq_filtered_exploded = mut_freq_filtered.explode('questions')

In [ ]:
mut_freq_filtered_exploded.shape

In [ ]:
mut_freq_filtered_exploded[mut_freq_filtered_exploded['questions'].str.strip() == ''].shape

In [ ]:
mut_freq_filtered_exploded= mut_freq_filtered_exploded[~(mut_freq_filtered_exploded['questions'].str.strip() == '')]

In [ ]:
mut_freq_filtered_exploded.shape

In [ ]:
mut_freq_filtered_exploded.head(n=3).iloc[0]['questions']

In [ ]:
### run above through gdc-qag v1 and only retain queries with frequency >= 5%
## output file ssms_gdc_ge_50.results.csv

In [ ]:
more_ssm_results = pd.read_csv('Data/ssms_gdc_ge_50.results.csv', index_col=0)

In [ ]:
more_ssm_results_ge_5 = more_ssm_results[more_ssm_results['ground_truth_stat'] >= 5]

In [ ]:
more_ssm_results_ge_5.shape

In [ ]:
more_ssm_results_ge_5['mutation_entities'].nunique()

### combination questions
- Run 18 questions in Table S2 through gdc-qag V1 and retain queries with frequency >= 5%
- N = 170
- results in combination_questions_final.results.csv

In [ ]:
combination_results = pd.read_csv('Data/combination_questions_final.results.csv', index_col=0)

In [ ]:
combination_results.shape

In [ ]:
combination_results[['questions', 'ground_truth_stat']].head(n=3)

## collate into final eval set

In [ ]:
eval_results = pd.concat([civic_results, more_ssm_results_ge_5])

In [ ]:
eval_results.shape

### count and remove rows that could not be processed through the pipeline
- e.g due to API unavailability or other issues in pipeline
- also, if there are any 0 freq ground-truth stat or freq < 5\%> (remove them)
- remove any duplicate questions (if there are still)

In [ ]:
eval_results[eval_results['final_response'].str.contains('unable to')]['intent'].unique()

In [ ]:
successful_results = eval_results[~eval_results['final_response'].str.contains('unable to')]

In [ ]:
successful_results[successful_results['ground_truth_stat']==0].shape

In [ ]:
successful_results[successful_results['ground_truth_stat']==0]['intent'].unique()

In [ ]:
successful_results[successful_results['ground_truth_stat']==0][['questions', 'intent']]

In [ ]:
successful_results = successful_results[successful_results['ground_truth_stat'] != 0]

In [ ]:
successful_results.shape

In [ ]:
successful_results.drop_duplicates(subset=['questions', 'list_cancer_entities'], keep='first', inplace=True)

In [ ]:
successful_results.shape

In [ ]:
# drop freq < 5%
a = successful_results['ground_truth_stat']< 5
successful_results = successful_results[~a]

In [ ]:
successful_results.shape

In [ ]:
successful_results['gene_entities'].nunique()

In [ ]:
successful_results['mutation_entities'].nunique()

In [ ]:
successful_results['mutation_entities'].unique()

In [ ]:
successful_results['list_cancer_entities'].nunique()

In [ ]:
eval_set = pd.concat([successful_results, combination_results])

In [ ]:
eval_set.shape

In [ ]:
eval_set.columns

In [ ]:
eval_set[['questions', 'ground_truth_stat']].head(n=3)

In [ ]:
# confirm groud_truth_stat for each query using entities, and running through independent GDC API functions
# see https://github.com/uc-cdis/gdc-qag/blob/master/check_6k_mutations_withTable2.py

In [ ]:
eval_set['gene_entities'].nunique()

In [ ]:
eval_set['gene_entities'].unique()

In [ ]:
eval_set['mutation_entities'].nunique()

In [ ]:
eval_set['mutation_entities'].unique()

In [ ]:
eval_set['cancer_entities'].nunique()

In [ ]:
eval_set['cancer_entities'].unique()